# 03 — OCR RTP vs TVI Comparison

Objetivo:

> Reproduzir e isolar a parte de comparação RTP vs TVI que estava no Notebook 2.

Este notebook usa exatamente a mesma lógica do Notebook 2 para:

1. preparar os dados OCR;
2. usar os mesmos dicionários de candidatos, partidos e temas;
3. calcular menções por telejornal;
4. comparar RTP vs TVI nas mesmas datas;
5. calcular diferenças RTP - TVI por tema e por candidato.

Este notebook é uma base para desenvolveres depois uma comparação mais profunda entre canais.


## 0. Preparação mínima dos dados

Carrega `ocr_long`, limpa o texto e cria as colunas necessárias:

```python
clean_text
tokens_content
n_words_content
```


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "features"
OUTPUT_DIR = BASE_DIR / "outputs_ocr_03"
OUTPUT_DIR.mkdir(exist_ok=True)

NORMALIZED_PATH = BASE_DIR / "outputs_ocr_01" / "ocr_long_normalized.csv"

print("BASE_DIR:", BASE_DIR.resolve())
print("FEATURES_DIR exists:", FEATURES_DIR.exists())
print("NORMALIZED_PATH exists:", NORMALIZED_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


# ---------- Loading helpers ----------

def parse_ocr_filename(path):
    path = Path(path)
    stem = path.stem.replace("_ocr", "")
    parts = stem.split("_")

    channel = parts[1] if len(parts) >= 2 else None
    month = parts[2] if len(parts) >= 3 else None
    day = parts[3] if len(parts) >= 4 else None

    return {
        "file": path.name,
        "channel": channel,
        "month": month,
        "day": day,
        "date_label": f"{month}_{day}" if month and day else None
    }

def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def extract_frame_number(frame_value, fallback_index):
    if isinstance(frame_value, (str, Path)):
        nums = re.findall(r"\d+", Path(str(frame_value)).stem)
        if nums:
            return int(nums[-1])
        return int(fallback_index)

    try:
        return int(float(frame_value))
    except Exception:
        return int(fallback_index)

def extract_text(det):
    if isinstance(det, dict):
        for key in ["text", "Text", "word", "value", "label"]:
            if key in det:
                return str(det[key])
    if isinstance(det, str):
        return det
    return None

def extract_conf(det):
    if isinstance(det, dict):
        for key in ["conf", "confidence", "score", "prob", "probability"]:
            if key in det:
                return safe_float(det[key])
    return np.nan

def normalize_ocr_df(df, file_name):
    meta = parse_ocr_filename(file_name)
    rows = []

    frame_col = "Frame" if "Frame" in df.columns else df.columns[0]
    ocr_col = "OCR" if "OCR" in df.columns else None

    if ocr_col is None:
        raise ValueError(f"Não encontrei coluna OCR em {file_name}. Colunas: {list(df.columns)}")

    for idx, row in df.iterrows():
        frame_original = row[frame_col]
        frame_number = extract_frame_number(frame_original, idx)
        detections = row[ocr_col]

        if not isinstance(detections, (list, tuple, np.ndarray)):
            continue

        for det in detections:
            text = extract_text(det)
            if text is None or len(str(text).strip()) == 0:
                continue

            rows.append({
                "file": file_name,
                "channel": meta["channel"],
                "date_label": meta["date_label"],
                "frame": frame_number,
                "frame_original": str(frame_original),
                "text": str(text),
                "confidence": extract_conf(det),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["second"] = out["frame"]
    out["minute"] = (out["second"] // 60).astype(int)
    return out


# ---------- Load normalized OCR ----------

if NORMALIZED_PATH.exists():
    ocr_long = pd.read_csv(NORMALIZED_PATH)
else:
    print("Normalized OCR not found. Loading raw OCR pickles...")
    ocr_files = sorted(FEATURES_DIR.glob("*_ocr.pkl"))
    parts = []

    for f in ocr_files:
        df = pd.read_pickle(f)
        norm = normalize_ocr_df(df, f.name)
        print(f.name, "->", len(norm), "detections")
        parts.append(norm)

    ocr_long = pd.concat(parts, ignore_index=True)

# Ensure expected types.
ocr_long["text"] = ocr_long["text"].astype(str)
ocr_long["frame"] = pd.to_numeric(ocr_long["frame"], errors="coerce")
ocr_long["minute"] = pd.to_numeric(ocr_long["minute"], errors="coerce").astype("Int64")


# ---------- Text cleaning helpers ----------

STOPWORDS_PT = {
    "de","a","o","e","que","do","da","em","um","uma","para","com","não","os","as","no","na","por",
    "se","ao","dos","das","mais","como","é","foi","são","ser","tem","também","ou","à","às","nos",
    "nas","sobre","entre","até","sem","já","lhe","ele","ela","eles","elas","sua","seu","suas","seus",
    "este","esta","estes","estas","isso","isto","há","vai","ter","mas","muito","muita","muitos","muitas",
    "porque","quando","onde","quem","qual","quais","todo","toda","todos","todas","num","numa","pelo","pela",
    "pelos","pelas","aos","ainda","só","era","foram","será","serem","nosso","nossa","seja","forma"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "nacional", "direto", "direita", "esquerda", "tvi", "rtp", "sic",
    "notícias", "noticia", "noticias", "edição", "especial", "última", "hora", "minuto",
    "portugal", "portuguesa", "português", "portugueses", "www", "pt"
}

def clean_text_pt(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-záàâãéèêíóôõúç0-9\s]", " ", text)
    text = re.sub(r"\b(b|br)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    tokens = clean_text_pt(text).split()
    tokens = [t for t in tokens if len(t) >= min_len]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens

ocr_long["clean_text"] = ocr_long["text"].apply(clean_text_pt)
ocr_long["tokens_content"] = ocr_long["clean_text"].apply(lambda x: tokenize_pt(x))
ocr_long["n_words_content"] = ocr_long["tokens_content"].apply(len)

print("ocr_long ready:", ocr_long.shape)
print("Files:", ocr_long["file"].nunique())
print("Channels:", sorted(ocr_long["channel"].dropna().unique()))
print("Dates:", sorted(ocr_long["date_label"].dropna().unique()))


## 1. Dicionário de candidatos, partidos e temas políticos

Esta secção foi copiada do Notebook 2 para garantir que a comparação RTP vs TVI usa exatamente os mesmos aliases.


In [ ]:
import re
import pandas as pd
from pathlib import Path

# candidate_party.pkl is used only as reference here.
candidate_party_path = FEATURES_DIR / "candidate_party.pkl"

if candidate_party_path.exists():
    candidate_party = pd.read_pickle(candidate_party_path)
    print("candidate_party:", candidate_party.shape)
    display(candidate_party)
else:
    candidate_party = None
    print("candidate_party.pkl não encontrado.")


# ------------------------------------------------------------
# Candidate aliases
# ------------------------------------------------------------
# Note:
# Some aliases are descriptive expressions used in TV news, not only names.
# Be careful with generic terms such as "seguro" or "almirante", which may create false positives.

CANDIDATE_ALIASES = {
    "André Ventura": [
        "andré ventura",
        "andre ventura",
        "ventura",
        "líder do chega",
        "lider do chega"
    ],

    "Cotrim Figueiredo": [
        "cotrim",
        "cotrim de figueiredo",
        "cotrim figueiredo",
        "joão cotrim de figueiredo",
        "joao cotrim de figueiredo"
    ],

    "Luís Marques Mendes": [
        "marques mendes",
        "luís marques mendes",
        "luis marques mendes"
    ],

    "Henrique Gouveia e Melo": [
        "gouveia e melo",
        "gouveia melo",
        "henrique gouveia e melo",
        "almirante gouveia e melo"
        # "almirante" can be added, but it is more generic.
    ],

    "António José Seguro": [
        "antónio josé seguro",
        "antonio jose seguro",
        "josé seguro",
        "jose seguro",
        "antónio seguro",
        "antonio seguro",
        # "seguro" alone is intentionally excluded because it can be a common word.
    ],

    "António Filipe": [
        "antónio filipe",
        "antonio filipe"
    ],

    "Catarina Martins": [
        "catarina martins",
        "ex coordenadora do bloco",
        "ex-coordenadora do bloco",
        "ex lider do bloco",
        "ex-lider do bloco"
    ],

    "Jorge Pinto": [
        "jorge pinto"
        # "pinto" alone is intentionally excluded because it can create false positives.
    ],
}


# ------------------------------------------------------------
# Party aliases
# ------------------------------------------------------------
# Short acronyms such as "PS", "IL", "BE" can create OCR false positives.
# We keep them, but interpret results carefully.

PARTY_ALIASES = {
    "CHEGA": [
        "chega",
        "partido chega"
    ],

    "IL": [
        "il",
        "iniciativa liberal",
        "liberais"
    ],

    "PSD": [
        "psd",
        "ad",
        "aliança democrática",
        "alianca democratica",
        "partido social democrata",
        "sociais democratas"
    ],

    "PS": [
        "ps",
        "partido socialista",
        "socialistas"
    ],

    "PCP/CDU": [
        "pcp",
        "cdu",
        "partido comunista",
        "comunistas"
    ],

    "BE": [
        "be",
        "bloco de esquerda",
    ],

    "LIVRE": [
        "livre",
        "partido livre"
    ],

    "CDS": [
        "cds",
        "cds pp",
        "cds-pp",
        "centro democrático social",
    ],
}


# ------------------------------------------------------------
# Topic/theme aliases
# ------------------------------------------------------------

THEME_ALIASES = {
    "Eleições/Campanha": [
        "presidenciais",
        "presidencial",
        "eleições",
        "eleicao",
        "eleições presidenciais",
        "eleicoes presidenciais",
        "candidato",
        "candidata",
        "candidatos",
        "candidatas",
        "candidatura",
        "candidaturas",
        "campanha",
        "abstenção",
        "abstencao",
        "urna",
        "urnas",
        "eleitor",
        "eleitores",
        "debate",
        "debates"
    ],

    "Sondagens": [
        "sondagem",
        "sondagens",
        "barómetro",
        "barometro",
        "voto",
        "votos",
        "intenção de voto",
        "intencao de voto",
        "intenções de voto",
        "intencoes de voto",
        "projecção",
        "projeccao",
        "projeção",
        "projecao"
    ],

    "Governo/Partidos": [
        "governo",
        "primeiro ministro",
        "primeira ministra",
        "montenegro",
        "luís montenegro",
        "luis montenegro",
        "parlamento",
        "assembleia",
        "partido",
        "partidos",
        "oposição",
        "oposicao"
    ],

    "Saúde": [
        "saúde",
        "saude",
        "sns",
        "hospital",
        "hospitais",
        "médico",
        "medico",
        "médicos",
        "medicos",
        "enfermeiro",
        "enfermeiros",
        "urgência",
        "urgencias",
        "urgência",
        "inem",
        "listas de espera",
        "lista de espera",
        "ambulância",
        "ambulancia"
    ],

    "Economia": [
        "economia",
        "inflação",
        "inflacao",
        "preços",
        "precos",
        "imposto",
        "impostos",
        "irs",
        "orçamento",
        "orcamento",
        "salário",
        "salario",
        "salários",
        "salarios",
        "rendimento",
        "pib",
        "juros",
        "bce"
    ],

    "Habitação": [
        "habitação",
        "habitacao",
        "casa",
        "casas",
        "arrendamento",
        "renda",
        "rendas",
        "senhorio",
        "senhorios",
        "inquilino",
        "inquilinos",
        "crédito habitação",
        "credito habitacao"
    ],

    "Educação": [
        "educação",
        "educacao",
        "escola",
        "escolas",
        "professor",
        "professores",
        "aluno",
        "alunos",
        "ensino",
        "aulas",
        "creche",
        "creches"
    ],

    "Justiça/Segurança": [
        "justiça",
        "justica",
        "tribunal",
        "tribunais",
        "polícia",
        "policia",
        "pj",
        "psp",
        "gnr",
        "segurança",
        "seguranca",
        "crime",
        "crimes",
        "corrupção",
        "corrupcao",
        "pgr",
        "ministério público",
        "ministerio publico"
    ],

    "Internacional": [
        "ucrânia",
        "ucrania",
        "rússia",
        "russia",
        "guerra",
        "israel",
        "gaza",
        "palestina",
        "trump",
        "eua",
        "estados unidos",
        "europa",
        "bruxelas",
        "união europeia",
        "uniao europeia"
    ],

    "Greves/Trabalho": [
        "greve",
        "greves",
        "sindicato",
        "sindicatos",
        "trabalhador",
        "trabalhadores",
        "protesto",
        "protestos",
        "manifestação",
        "manifestacao",
        "manifestantes",
        "patrões",
        "patroes"
    ],
}


# ------------------------------------------------------------
# Alias matching functions
# ------------------------------------------------------------

def normalize_alias(alias):
    """
    Applies the same text normalization used for OCR text.
    """
    return clean_text_pt(alias)


def contains_alias(text, aliases):
    """
    Returns True if any alias appears in the OCR text.

    Uses word-boundary-like regex to avoid partial matches.
    Example: "ps" should not match inside another longer word.
    """
    if not isinstance(text, str):
        return False

    text = clean_text_pt(text)

    for alias in aliases:
        alias = normalize_alias(alias)

        if not alias:
            continue

        pattern = r"(?<!\w)" + re.escape(alias) + r"(?!\w)"

        if re.search(pattern, text):
            return True

    return False


print("N candidates:", len(CANDIDATE_ALIASES))
print("N parties:", len(PARTY_ALIASES))
print("N themes:", len(THEME_ALIASES))

## 2. Menções por telejornal

Esta secção calcula as tabelas necessárias para comparar RTP vs TVI.

Output principal:

```python
theme_by_file
candidate_by_file
party_by_file
```


In [ ]:
def mention_table_by_file(df, alias_dict, entity_type):
    rows = []

    for (file, channel, date_label), group in df.groupby(["file", "channel", "date_label"]):
        total_detections = len(group)
        total_words = group["n_words_content"].sum()

        for label, aliases in alias_dict.items():
            mask = group["clean_text"].apply(lambda t: contains_alias(t, aliases))
            detections = int(mask.sum())

            rows.append({
                "file": file,
                "channel": channel,
                "date_label": date_label,
                "entity_type": entity_type,
                "entity": label,
                "detections": detections,
                "per_1000_detections": 1000 * detections / max(total_detections, 1),
                "per_1000_words": 1000 * detections / max(total_words, 1),
            })

    return pd.DataFrame(rows)

candidate_by_file = mention_table_by_file(ocr_long, CANDIDATE_ALIASES, "candidate")
party_by_file = mention_table_by_file(ocr_long, PARTY_ALIASES, "party")
theme_by_file = mention_table_by_file(ocr_long, THEME_ALIASES, "theme")

display(theme_by_file.sort_values("detections", ascending=False).head(30))
display(candidate_by_file.sort_values("detections", ascending=False).head(30))

candidate_by_file.to_csv(OUTPUT_DIR / "candidate_mentions_by_file.csv", index=False)
party_by_file.to_csv(OUTPUT_DIR / "party_mentions_by_file.csv", index=False)
theme_by_file.to_csv(OUTPUT_DIR / "theme_mentions_by_file.csv", index=False)


## 8. RTP vs TVI nas mesmas datas

Comparação direta entre canais para as mesmas datas.

A métrica principal é normalizada por 1000 palavras OCR para evitar que telejornais com mais texto ganhem sempre.


In [ ]:
# ============================================================
# 8. RTP vs TVI — comparação no mesmo dia
# ============================================================

MONTH_ORDER = {
    "Nov": 1,
    "Dec": 2,
    "Jan": 3
}

CHANNEL_ORDER = ["RTP", "TVI"]

def date_label_to_sort(date_label):
    month, day = str(date_label).split("_")
    return MONTH_ORDER.get(month, 999) * 100 + int(day)


# Ficheiros suspeitos / contaminados
SUSPECT_FILES = [
    "Telejornal_TVI_Dec_2_ocr.pkl"
]

# Filtrar tabelas já calculadas
theme_by_file_analysis = theme_by_file.copy()
candidate_by_file_analysis = candidate_by_file.copy()

if "file" in theme_by_file_analysis.columns:
    theme_by_file_analysis = theme_by_file_analysis[
        ~theme_by_file_analysis["file"].isin(SUSPECT_FILES)
    ].copy()

if "file" in candidate_by_file_analysis.columns:
    candidate_by_file_analysis = candidate_by_file_analysis[
        ~candidate_by_file_analysis["file"].isin(SUSPECT_FILES)
    ].copy()


# Temas: RTP vs TVI por data
theme_channel_date = (
    theme_by_file_analysis
    .groupby(["date_label", "channel", "entity"])
    .agg(
        detections=("detections", "sum"),
        per_1000_detections=("per_1000_detections", "sum"),
        per_1000_words=("per_1000_words", "sum")
    )
    .reset_index()
)

# Candidatos: RTP vs TVI por data
candidate_channel_date = (
    candidate_by_file_analysis
    .groupby(["date_label", "channel", "entity"])
    .agg(
        detections=("detections", "sum"),
        per_1000_detections=("per_1000_detections", "sum"),
        per_1000_words=("per_1000_words", "sum")
    )
    .reset_index()
)

display(theme_channel_date.head())
display(candidate_channel_date.head())

In [ ]:
# Escolher apenas datas onde existem os dois canais: RTP e TVI

date_channel_counts = (
    theme_channel_date[["date_label", "channel"]]
    .drop_duplicates()
    .groupby("date_label")["channel"]
    .nunique()
)

paired_dates = date_channel_counts[date_channel_counts == 2].index.tolist()
paired_dates = sorted(paired_dates, key=date_label_to_sort)

print("Datas com RTP e TVI disponíveis:")
print(paired_dates)

chosen_date = paired_dates[0]

print("chosen_date =", chosen_date)

In [ ]:
def plot_rtp_vs_tvi_for_date(table, chosen_date, title_prefix, value_col="per_1000_words"):
    data_date = table[table["date_label"] == chosen_date].copy()

    pivot = data_date.pivot_table(
        index="entity",
        columns="channel",
        values=value_col,
        aggfunc="sum",
        fill_value=0
    )

    # Garantir sempre ordem RTP, TVI
    pivot = pivot.reindex(columns=CHANNEL_ORDER, fill_value=0)

    # Remover linhas com zero nos dois canais
    pivot = pivot[pivot.sum(axis=1) > 0]

    # Ordenar por presença total
    pivot["total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("total", ascending=False).drop(columns="total")

    display(pivot)

    pivot.plot(kind="bar", figsize=(11, 4))
    plt.title(f"{title_prefix} — {chosen_date}")
    plt.ylabel("Menções por 1000 palavras OCR")
    plt.xlabel("")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return pivot


pivot_theme_date = plot_rtp_vs_tvi_for_date(
    theme_channel_date,
    chosen_date=chosen_date,
    title_prefix="RTP vs TVI — temas no OCR"
)

pivot_candidate_date = plot_rtp_vs_tvi_for_date(
    candidate_channel_date,
    chosen_date=chosen_date,
    title_prefix="RTP vs TVI — candidatos no OCR"
)

In [ ]:
for date in paired_dates:
    print("=" * 80)
    print("Data:", date)

    plot_rtp_vs_tvi_for_date(
        theme_channel_date,
        chosen_date=date,
        title_prefix="RTP vs TVI — temas no OCR"
    )

    plot_rtp_vs_tvi_for_date(
        candidate_channel_date,
        chosen_date=date,
        title_prefix="RTP vs TVI — candidatos no OCR"
    )

## 9. Diferenças RTP - TVI por tema/candidato

Valores positivos indicam maior presença normalizada na RTP.  
Valores negativos indicam maior presença normalizada na TVI.


In [ ]:
def channel_difference_table(channel_date_table):
    pivot = channel_date_table.pivot_table(
        index=["date_label", "entity"],
        columns="channel",
        values="per_1000_words",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    if "RTP" not in pivot.columns:
        pivot["RTP"] = 0
    if "TVI" not in pivot.columns:
        pivot["TVI"] = 0

    pivot["RTP_minus_TVI"] = pivot["RTP"] - pivot["TVI"]
    pivot["abs_difference"] = pivot["RTP_minus_TVI"].abs()
    return pivot.sort_values("abs_difference", ascending=False)

theme_diff = channel_difference_table(theme_channel_date)
candidate_diff = channel_difference_table(candidate_channel_date)

display(Markdown("### Maiores diferenças por tema"))
display(theme_diff.head(30))

display(Markdown("### Maiores diferenças por candidato"))
display(candidate_diff.head(30))

theme_diff.to_csv(OUTPUT_DIR / "rtp_tvi_theme_differences.csv", index=False)
candidate_diff.to_csv(OUTPUT_DIR / "rtp_tvi_candidate_differences.csv", index=False)


## 5. Outputs gerados

Este notebook guarda outputs em:

```text
outputs_ocr_03/
```

Principais outputs:

- `candidate_mentions_by_file.csv`
- `party_mentions_by_file.csv`
- `theme_mentions_by_file.csv`
- `rtp_tvi_theme_differences.csv`
- `rtp_tvi_candidate_differences.csv`

Nota: este notebook ainda é uma extração direta da parte RTP vs TVI do Notebook 2.  
Depois podes aprofundar com rankings, seleção de datas específicas, gráficos mais limpos e métricas adicionais.
